In [ ]:
#READ EXCEL
import pandas as pd
import numpy as np

df = pd.read_excel("Cleaned_FIFA_Data.xlsx")

print(df.head())


In [ ]:
# GET UNIQUE CLUBS
# #clubs = df["Clubs"]
#print(clubs.head())

unique_clubs = df["Club"].unique()

print(np.sort(unique_clubs))


[' SSV Jahn Regensburg' '1. FC Heidenheim' '1. FC Kaiserslautern'
 '1. FC Köln' '1. FC Magdeburg' '1. FC Nürnberg' '1. FC Union Berlin'
 '1. FSV Mainz 05' 'AC Ajaccio' 'AC Horsens' 'AD Alcorcón' 'ADO Den Haag'
 'AEK Athens' 'AFC Eskilstuna' 'AFC Wimbledon' 'AIK Solna' 'AJ Auxerre'
 'AS Monaco' 'AS Nancy Lorraine' 'AS Saint-Étienne' 'AZ Alkmaar'
 'Aalborg BK' 'Aalesunds FK' 'Aarhus GF' 'Aberdeen' 'Accrington Stanley'
 'Adelaide United' 'Ajax' 'Akhisar Belediyespor' 'Al Ahli' 'Al Batin'
 'Al Faisaly' 'Al Fateh' 'Al Fayha' 'Al Hilal' 'Al Ittihad' 'Al Nassr'
 'Al Qadisiyah' 'Al Raed' 'Al Shabab' 'Al Taawoun' 'Alanyaspor'
 'Albacete Balompié' 'Albirex Niigata' 'Alianza Petrolera'
 'Amiens SC Football' 'Amkar Perm' 'Angers SCO' 'Antalyaspor'
 'Argentinos Juniors' 'Arka Gdynia' 'Arsenal' 'Arsenal Tula'
 'Arsenal de Sarandí' 'Ascoli' 'Asociacion Deportivo Cali'
 'Associação Atlética Ponte Preta' 'Associação Chapecoense de Futebol'
 'Aston Villa' 'Atalanta' 'Athletic Club de Bilbao' 'Atiker Kon

In [ ]:
# ENTER UNIQUE CLUBS
clubs_df = pd.DataFrame(np.sort(unique_clubs), columns= ["Club"])
clubs_df.dropna()
clubs_df.drop_duplicates()

print(clubs_df) 

                       Club
0       SSV Jahn Regensburg
1          1. FC Heidenheim
2      1. FC Kaiserslautern
3                1. FC Köln
4           1. FC Magdeburg
..                      ...
643         Águilas Doradas
644               Örebro SK
645           Östersunds FK
646  İstanbul Başakşehir FK
647           Śląsk Wrocław

[648 rows x 1 columns]


In [ ]:
# FIX CLUB NAMES
clubs_df["Fixed_Name"] = clubs_df["Fixed_Name"].str.replace( "CD ", "Club Deportivo ")

clubs_df["Search_Name"] = (clubs_df["Fixed_Name"].str.replace(' ', '_'))

clubs_df["Crest_URL"] = ""

print(clubs_df)

                       Club              Fixed_Name             Search_Name  \
0       SSV Jahn Regensburg     SSV Jahn Regensburg    _SSV_Jahn_Regensburg   
1          1. FC Heidenheim        1. FC Heidenheim        1._FC_Heidenheim   
2      1. FC Kaiserslautern    1. FC Kaiserslautern    1._FC_Kaiserslautern   
3                1. FC Köln              1. FC Köln              1._FC_Köln   
4           1. FC Magdeburg         1. FC Magdeburg         1._FC_Magdeburg   
..                      ...                     ...                     ...   
643         Águilas Doradas         Águilas Doradas         Águilas_Doradas   
644               Örebro SK               Örebro SK               Örebro_SK   
645           Östersunds FK           Östersunds FK           Östersunds_FK   
646  İstanbul Başakşehir FK  İstanbul Başakşehir FK  İstanbul_Başakşehir_FK   
647           Śląsk Wrocław           Śląsk Wrocław           Śląsk_Wrocław   

    Club_URL Crest_URL  
0                       
1

In [ ]:
# TEST GET URL
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Granada_CF"

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

response = requests.get(url, headers=my_info_header)
#print(response.status_code)
#print(response.text)

soup = BeautifulSoup(response.text, "html.parser")

#print ("*****")
#print(soup.title)

infobox = soup.find("table", class_="infobox")

img = infobox.find("img")

crest_url = img["src"]

if crest_url.startswith("//"):
    crest_url = "https:" + crest_url

print(crest_url)


https://upload.wikimedia.org/wikipedia/en/thumb/d/d5/Logo_of_Granada_Club_de_F%C3%BAtbol.svg/120px-Logo_of_Granada_Club_de_F%C3%BAtbol.svg.png


In [ ]:
#TEST DOWNLOAD IMAGE
my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

img_response = requests.get(crest_url, headers=my_info_header)

print(img_response.status_code)
print(img_response.headers["Content-Type"])

with open("assets/crest_images/_crest.png", "wb") as f:
    f.write(img_response.content)


200
image/png


In [47]:
# FUNCTIONS GET URL N DOWNLOAD IMAGE
import requests
from bs4 import BeautifulSoup

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

def get_crest_url(club):
        # CONNECT
        url = f"https://en.wikipedia.org/wiki/{club}"

        response = requests.get(url, headers=my_info_header)

        soup = BeautifulSoup(response.text, "html.parser")

        infobox = soup.find("table", class_="infobox")
        if infobox is None:
            print(f"No infobox: {club}")
            return None

        # GET IMAGE BLOCK IN INFOBOX
        img = infobox.find("img")
        if img is None:
            print(f"No image: {club}")
            return None

        # FIND SOURCE IMAGE, NOT HREF TO WEB
        crest_url = img["src"]
        if not crest_url:
            return None
        
        if crest_url.startswith("//"):
            crest_url = "https:" + crest_url

        print(crest_url)
        return crest_url

def download_crest_image(url, club):
        # INVALID LINK
        if pd.isna(url) or not url:
            url= "https://upload.wikimedia.org/wikipedia/commons/thumb/2/21/Solid_black.svg/500px-Solid_black.svg.png"
       
        # DOWNLOAD IMAGES            
        img_response = requests.get(url, headers=my_info_header)

        print(img_response.status_code)
        print(img_response.headers["Content-Type"])

        with open(f"assets/crest_images/{club}_crest.png", "wb") as f:
            f.write(img_response.content)



In [ ]:
# USE GET URL
clubs_df["Crest_URL"] = clubs_df["Search_Name"].apply(get_crest_url)

No infobox: Aalborg_BK
No infobox: Al_Ahli
No infobox: Al_Batin
No infobox: Al_Faisaly
No image: Al_Fateh
No infobox: Al_Fayha
No infobox: Al_Hilal
No infobox: Al_Ittihad
No infobox: Al_Qadisiyah
No infobox: Al_Shabab
No infobox: Al_Taawoun
No infobox: Amiens_SC_Football
No infobox: Arsenal
No infobox: Ascoli
No infobox: Asociacion_Deportivo_Cali
No infobox: Atletico_Nacional_Medellin
No infobox: Banfield
No infobox: Barnet
No infobox: Bourg-en-Bresse_Péronnas_01
No infobox: Bury
No infobox: Club_Deportivo_America_de_Cali
No infobox: Club_Deportivo_Antofagasta
No infobox: Club_Deportivo_Aves
No infobox: Club_Deportivo_Feirense
No infobox: Club_Deportivo_Los_Millionarios_Bogota
No infobox: Club_Deportivo_Once_Caldas_Manizales
No infobox: CPD_Junior_Barranquilla
No infobox: Carpi
No infobox: Chelsea
No infobox: Chesterfield
No infobox: Cracovia
No infobox: Cruzeiro
No infobox: Crystal_Palace
No infobox: Derry_City
No infobox: Everton
No infobox: Evkur_Yeni_Malatyaspor
No infobox: Excelsi

In [ ]:
# FUNCTION TO GET CLOSE NAME
import requests

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

def wikipedia_search(club):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": club + " football club",
        "format": "json",
        "srlimit": 5
    }

    r = requests.get(url, params=params, headers=my_info_header).json()

    results = r["query"]["search"]

    if results:
        return results[0]["title"]

    return None


In [ ]:
#RESULTS OF WIKIPEDIA SEARCH
import time

results = []

for club in clubs_df["Club"]:
    result = wikipedia_search(club)
    print(club, "→", result)
    results.append(result)
    time.sleep(0.5)

clubs_df["Fixed_Name"] = results

 SSV Jahn Regensburg → SSV Jahn Regensburg
1. FC Heidenheim → 1. FC Heidenheim
1. FC Kaiserslautern → 1. FC Kaiserslautern
1. FC Köln → 1. FC Köln
1. FC Magdeburg → 1. FC Magdeburg
1. FC Nürnberg → 1. FC Nürnberg
1. FC Union Berlin → 1. FC Union Berlin
1. FSV Mainz 05 → 1. FSV Mainz 05
AC Ajaccio → AC Ajaccio
AC Horsens → AC Horsens
AD Alcorcón → AD Alcorcón
ADO Den Haag → ADO Den Haag Stadium
AEK Athens → AEK Athens F.C.
AFC Eskilstuna → AFC Eskilstuna
AFC Wimbledon → AFC Wimbledon
AIK Solna → AIK
AJ Auxerre → AJ Auxerre
AS Monaco → AS Monaco FC
AS Nancy Lorraine → AS Nancy Lorraine
AS Saint-Étienne → AS Saint-Étienne
AZ Alkmaar → AZ Alkmaar
Aalborg BK → AaB Fodbold
Aalesunds FK → Aalesunds FK
Aarhus GF → Aarhus Gymnastikforening
Aberdeen → Aberdeen F.C.
Accrington Stanley → Accrington Stanley F.C.
Adelaide United → Adelaide United FC
Ajax → AFC Ajax
Akhisar Belediyespor → Luciano Guaycochea
Al Ahli → Shabab Al Ahli Club
Al Batin → Al Batin FC
Al Faisaly → Al-Faisaly SC
Al Fateh → Al 

In [ ]:
# EXPORT DATA URL TO CSV

clubs_df["Search_Name"] = (clubs_df["Fixed_Name"].str.replace(' ', '_'))
print(clubs_df)


#clubs_df.to_csv("club_crest_urls.csv", index=True)

                       Club                Fixed_Name  \
0       SSV Jahn Regensburg       SSV Jahn Regensburg   
1          1. FC Heidenheim          1. FC Heidenheim   
2      1. FC Kaiserslautern      1. FC Kaiserslautern   
3                1. FC Köln                1. FC Köln   
4           1. FC Magdeburg           1. FC Magdeburg   
..                      ...                       ...   
643         Águilas Doradas           Águilas Doradas   
644               Örebro SK                 Örebro SK   
645           Östersunds FK             Östersunds FK   
646  İstanbul Başakşehir FK  İstanbul Başakşehir F.K.   
647           Śląsk Wrocław             Śląsk Wrocław   

                  Search_Name Club_URL Crest_URL  
0         SSV_Jahn_Regensburg                     
1            1._FC_Heidenheim                     
2        1._FC_Kaiserslautern                     
3                  1._FC_Köln                     
4             1._FC_Magdeburg                     
..       

In [ ]:
#GET URLS
clubs_df["Crest_URL"] = clubs_df["Search_Name"].apply(get_crest_url)

https://upload.wikimedia.org/wikipedia/commons/thumb/3/3d/Jahn_Regensburg_logo2014.svg/250px-Jahn_Regensburg_logo2014.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/9/9d/1._FC_Heidenheim_1846.svg/250px-1._FC_Heidenheim_1846.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/Logo_1_FC_Kaiserslautern.svg/250px-Logo_1_FC_Kaiserslautern.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/0/01/1._FC_Koeln_Logo_2014%E2%80%93.svg/250px-1._FC_Koeln_Logo_2014%E2%80%93.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/8/84/1._FC_Magdeburg.svg/250px-1._FC_Magdeburg.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/1._FC_N%C3%BCrnberg_logo.svg/250px-1._FC_N%C3%BCrnberg_logo.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/4/44/1._FC_Union_Berlin_Logo.svg/330px-1._FC_Union_Berlin_Logo.svg.png
https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/1._FSV_Mainz_05_logo.svg/250px-1._FSV_Mainz_05_logo.svg.png
https://upload

In [ ]:
#MAKE FINAL CSV
print(clubs_df)

clubs_df.to_csv("club_crest_urls.csv", index=True)

                       Club                Fixed_Name  \
0       SSV Jahn Regensburg       SSV Jahn Regensburg   
1          1. FC Heidenheim          1. FC Heidenheim   
2      1. FC Kaiserslautern      1. FC Kaiserslautern   
3                1. FC Köln                1. FC Köln   
4           1. FC Magdeburg           1. FC Magdeburg   
..                      ...                       ...   
643         Águilas Doradas           Águilas Doradas   
644               Örebro SK                 Örebro SK   
645           Östersunds FK             Östersunds FK   
646  İstanbul Başakşehir FK  İstanbul Başakşehir F.K.   
647           Śląsk Wrocław             Śląsk Wrocław   

                  Search_Name Club_URL  \
0         SSV_Jahn_Regensburg            
1            1._FC_Heidenheim            
2        1._FC_Kaiserslautern            
3                  1._FC_Köln            
4             1._FC_Magdeburg            
..                        ...      ...   
643           Águilas

In [ ]:
# LOAD FIXED CSV N DOWNLOAD IMAGES
import pandas as pd

clubs_df.apply(
    lambda rec: download_crest_image(rec["Crest_URL"], rec["Search_Name"]),
    axis=1
)

200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
image/png
200
im

0      None
1      None
2      None
3      None
4      None
       ... 
159    None
160    None
161    None
162    None
163    None
Length: 164, dtype: object

In [ ]:
# LOAD FIXED CSV N DOWNLOAD IMAGES

clubs_fixed = pd.read_csv("club_crest_urls.csv")

clubs_fixed.apply(
    lambda rec: download_crest_image( rec["Crest_URL"], rec["Search_Name"]),
    axis=1
)

NameError: name 'download_crest_image' is not defined